<div style="background: linear-gradient(135deg, #2C5F2D 0%, #5A8A3C 100%); padding: 32px 36px; border-radius: 12px; margin-bottom: 8px;">
  <h1 style="color: white; margin: 0 0 6px 0; font-size: 28px;">🌿 Preprocessing Dataset Klasifikasi Daun</h1>
  <p style="color: #d4edda; margin: 0; font-size: 15px;">Pipeline lengkap: cleaning · standarisasi · augmentasi</p>
</div>

<table style="width:100%; border-collapse:collapse; margin-top:12px; font-size:13px;">
<tr>
  <td style="padding:6px 12px; background:#f0f7f0; border:1px solid #c3dfc3; border-radius:4px; width:25%"><b>👤 Nama</b><br><span style="color:#555">Damarjati Suryo Laksono</span></td>
  <td style="padding:6px 12px; background:#f0f7f0; border:1px solid #c3dfc3; border-radius:4px; width:25%"><b>📅 Tanggal</b><br><span style="color:#555">6 Maret 2026</span></td>
  <td style="padding:6px 12px; background:#f0f7f0; border:1px solid #c3dfc3; border-radius:4px; width:25%"><b>📁 Dataset</b><br><span style="color:#555">daun_bagus · daun_layu</span></td>
  <td style="padding:6px 12px; background:#f0f7f0; border:1px solid #c3dfc3; border-radius:4px; width:25%"><b>🔧 Framework</b><br><span style="color:#555">Python · PIL · NumPy</span></td>
</tr>
</table>


---
## 📋 Daftar Isi

| # | Tahap | Deskripsi |
|---|-------|-----------|
| 1 | ⚙️ **Setup & Konfigurasi** | Mount Google Drive, import library, set parameter |
| 2 | 📂 **Analisis Dataset Awal** | Cek jumlah gambar & keseimbangan kelas |
| 3 | 🔍 **Deteksi Duplikat** | MD5 hash + perceptual hash |
| 4 | 🧹 **Filter Kualitas** | Hapus blur, gelap, terang, rusak |
| 5 | 🖼️ **Standarisasi** | Resize 224×224, RGB, normalisasi [0,1] |
| 6 | 🔄 **Augmentasi** | ×6 per gambar, 8 teknik |
| 7 | 📊 **Laporan Visual** | Grafik statistik hasil preprocessing |

---


## ⚙️ Step 1 — Setup & Konfigurasi

> Mount Google Drive dan definisikan semua parameter preprocessing.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os, sys, shutil, hashlib, warnings
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from PIL import Image, ImageStat, ImageEnhance
from collections import defaultdict

warnings.filterwarnings("ignore")

# ════════════════════════════════════════════════════
# KONFIGURASI — sesuaikan sebelum menjalankan pipeline
# ════════════════════════════════════════════════════

DATASET_DIR    = "/content/drive/MyDrive/Dataset_redi"  # ← ubah ke path dataset kamu
PROCESSED_DIR  = "dataset_processed"   # output setelah cleaning
AUGMENTED_DIR  = "dataset_augmented"   # output setelah augmentasi
CLASSES        = ["daun_bagus", "daun_layu"]

# Standarisasi
TARGET_SIZE    = (224, 224)   # ukuran gambar output
COLOR_MODE     = "RGB"        # "RGB" atau "L" (grayscale)
NORM_RANGE     = (0, 1)       # rentang normalisasi
TARGET_FORMAT  = ".jpg"       # format output
SUPPORTED_FORMATS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tiff"}

# Threshold kualitas
BLUR_THRESHOLD   = 5.0    # Laplacian variance < nilai ini → blur berat
DARK_THRESHOLD   = 10.0   # mean brightness < nilai ini → terlalu gelap
BRIGHT_THRESHOLD = 240.0  # mean brightness > nilai ini → terlalu terang
MIN_FILE_SIZE_KB = 2       # ukuran file minimum (KB)

# Augmentasi
AUG_PER_IMAGE    = 6       # jumlah augmentasi per gambar
ROTATION_RANGE   = 20      # derajat rotasi maksimum
ZOOM_RANGE       = 0.15    # faktor zoom (15%)
BRIGHTNESS_RANGE = (0.7, 1.3)
CONTRAST_RANGE   = (0.7, 1.3)
CROP_FRACTION    = 0.1     # potong 10% tepi

print("✅ Konfigurasi berhasil dimuat.")
print(f"   Dataset   : {DATASET_DIR}")
print(f"   Kelas     : {CLASSES}")
print(f"   Target    : {TARGET_SIZE}, {COLOR_MODE}, norm={NORM_RANGE}")
print(f"   Aug/gambar: {AUG_PER_IMAGE}×")


### 🛠️ Fungsi Utilitas

Fungsi-fungsi helper yang digunakan di seluruh pipeline.


In [ ]:
def image_hash(img_path: Path) -> str:
    """MD5 hash dari konten file untuk deteksi duplikat persis."""
    with open(img_path, "rb") as f:
        return hashlib.md5(f.read()).hexdigest()


def perceptual_hash(img: Image.Image, size: int = 8) -> str:
    """Perceptual hash untuk mendeteksi gambar yang nyaris sama."""
    small  = img.convert("L").resize((size, size), Image.LANCZOS)
    pixels = np.array(small).flatten()
    avg    = pixels.mean()
    return "".join("1" if p > avg else "0" for p in pixels)


def hamming_distance(h1: str, h2: str) -> int:
    """Jarak Hamming antara dua hash string."""
    return sum(c1 != c2 for c1, c2 in zip(h1, h2))


def laplacian_variance(img: Image.Image) -> float:
    """Nilai variance Laplacian — semakin rendah semakin blur."""
    from scipy.ndimage import convolve
    gray   = np.array(img.convert("L"), dtype=np.float32)
    kernel = np.array([[0, 1, 0], [1, -4, 1], [0, 1, 0]], dtype=np.float32)
    return float(np.var(convolve(gray, kernel)))


def mean_brightness(img: Image.Image) -> float:
    """Rata-rata kecerahan gambar (skala 0–255)."""
    return ImageStat.Stat(img.convert("L")).mean[0]


def is_corrupted(path: Path) -> bool:
    """Cek apakah file gambar rusak/tidak bisa dibuka."""
    try:
        with Image.open(path) as img:
            img.verify()
        return False
    except Exception:
        return True

print("✅ Fungsi utilitas berhasil didefinisikan.")


---
## 📂 Step 2 — Analisis Dataset Awal

Hitung jumlah gambar per kelas dan periksa keseimbangan dataset.


In [ ]:
def check_dataset(dataset_dir: str) -> dict:
    """Analisis awal jumlah gambar dan keseimbangan kelas."""
    stats = {}
    print("=" * 55)
    print("  STEP 2: ANALISIS DATASET AWAL")
    print("=" * 55)

    for cls in CLASSES:
        cls_path = Path(dataset_dir) / cls
        if not cls_path.exists():
            print(f"  ⚠️  Folder '{cls}' tidak ditemukan → membuat folder kosong")
            cls_path.mkdir(parents=True, exist_ok=True)

        images = [p for p in cls_path.iterdir()
                  if p.suffix.lower() in SUPPORTED_FORMATS]
        stats[cls] = {"total": len(images), "paths": images}
        print(f"  📁 Kelas '{cls}': {len(images)} gambar")

    # Cek balance
    counts = [stats[c]["total"] for c in CLASSES]
    if len(counts) >= 2:
        ratio = max(counts) / (min(counts) + 1e-9)
        if ratio > 1.5:
            print(f"  ⚠️  Dataset TIDAK SEIMBANG (rasio {ratio:.2f}×) → pertimbangkan over/undersampling")
        else:
            print(f"  ✅ Dataset cukup seimbang (rasio {ratio:.2f}×)")

    print(f"  ✅ Jumlah kelas: {len(CLASSES)} → memenuhi syarat minimal 2 kelas")
    return stats


init_stats = check_dataset(DATASET_DIR)


---
## 🔍 Step 3 — Deteksi Duplikat

Mendeteksi gambar duplikat menggunakan dua metode:
- **MD5 Hash** → duplikat persis (byte-by-byte identical)
- **Perceptual Hash** → hampir sama (hamming distance ≤ 5)


In [ ]:
def detect_duplicates(stats: dict) -> dict:
    """Deteksi duplikat persis (MD5) dan hampir sama (perceptual hash)."""
    print("=" * 55)
    print("  STEP 3: DETEKSI DUPLIKAT")
    print("=" * 55)

    report = {}
    for cls in CLASSES:
        paths       = stats[cls]["paths"]
        seen_md5    = {}
        seen_phash  = {}
        duplicates  = []
        near_dupes  = []

        for p in paths:
            if is_corrupted(p):
                continue

            # MD5 — duplikat persis
            h = image_hash(p)
            if h in seen_md5:
                duplicates.append((p, seen_md5[h]))
                print(f"  🔴 [DUPLIKAT PERSIS] {p.name} == {seen_md5[h].name}")
            else:
                seen_md5[h] = p

            # Perceptual hash — hampir sama
            try:
                with Image.open(p) as img:
                    ph    = perceptual_hash(img)
                    close = [seen_phash[k] for k in seen_phash
                             if hamming_distance(ph, k) <= 5]
                    if close:
                        near_dupes.append((p, close[0]))
                        print(f"  🟡 [HAMPIR SAMA]    {p.name} ≈ {close[0].name}")
                    else:
                        seen_phash[ph] = p
            except Exception:
                pass

        report[cls] = {"exact_duplicates": duplicates, "near_duplicates": near_dupes}
        print(f"  📊 Kelas '{cls}': {len(duplicates)} duplikat persis, {len(near_dupes)} hampir sama")

    return report


dup_report = detect_duplicates(init_stats)


---
## 🧹 Step 4 — Filter Kualitas Gambar

Menyaring gambar bermasalah berdasarkan:

| Kriteria | Metode | Threshold |
|----------|--------|-----------|
| **Blur berat** | Laplacian variance | `< 5.0` |
| **Terlalu gelap** | Mean brightness | `< 10.0` |
| **Terlalu terang** | Mean brightness | `> 240.0` |
| **File rusak** | PIL verify | exception |
| **File kosong** | Ukuran file | `< 2 KB` |


In [ ]:
def filter_quality(stats: dict) -> dict:
    """Filter gambar bermasalah: blur, gelap, terang, rusak."""
    print("=" * 55)
    print("  STEP 4: FILTER KUALITAS GAMBAR")
    print("=" * 55)

    # Cek ketersediaan scipy
    try:
        from scipy.ndimage import convolve as _
        use_blur = True
    except ImportError:
        print("  ⚠️  scipy tidak tersedia → melewati deteksi blur")
        use_blur = False

    quality_report = {}
    for cls in CLASSES:
        paths  = stats[cls]["paths"]
        issues = []

        for p in paths:
            # Cek ukuran file
            if p.stat().st_size < MIN_FILE_SIZE_KB * 1024:
                issues.append((p, "file terlalu kecil / rusak"))
                continue
            # Cek korupsi file
            if is_corrupted(p):
                issues.append((p, "file rusak/korup"))
                continue
            try:
                with Image.open(p) as img:
                    img.load()
                    brightness = mean_brightness(img)
                    if brightness < DARK_THRESHOLD:
                        issues.append((p, f"terlalu gelap (brightness={brightness:.1f})"))
                    elif brightness > BRIGHT_THRESHOLD:
                        issues.append((p, f"terlalu terang (brightness={brightness:.1f})"))
                    elif use_blur:
                        blur_val = laplacian_variance(img)
                        if blur_val < BLUR_THRESHOLD:
                            issues.append((p, f"blur berat (laplacian var={blur_val:.2f})"))
            except Exception as e:
                issues.append((p, f"error baca gambar: {e}"))

        quality_report[cls] = issues

        if issues:
            print(f"  ⚠️  Kelas '{cls}': {len(issues)} gambar bermasalah:")
            for p, reason in issues:
                print(f"      - {p.name}: {reason}")
        else:
            print(f"  ✅ Kelas '{cls}': semua gambar OK")

    return quality_report


quality_report = filter_quality(init_stats)


  STEP 4: FILTER KUALITAS GAMBAR
  ⚠️  Kelas 'daun_bagus': 8 gambar bermasalah:
      - IMG_20260306_152741_341.jpg: blur berat (laplacian var=4.19)
      - IMG_20260306_152725_074.jpg: blur berat (laplacian var=3.53)
      - IMG_20260306_152752_626.jpg: blur berat (laplacian var=4.36)
      - IMG_20260306_152746_784.jpg: blur berat (laplacian var=3.47)
      - IMG_20260306_154611_802.jpg: blur berat (laplacian var=4.20)
      - IMG_20260306_154622_373.jpg: blur berat (laplacian var=4.19)
      - IMG_20260306_154616_133.jpg: blur berat (laplacian var=3.79)
      - IMG_20260306_154618_913.jpg: blur berat (laplacian var=4.75)
  ⚠️  Kelas 'daun_layu': 16 gambar bermasalah:
      - IMG_20260306_151951_868.jpg: terlalu gelap (brightness=8.3)
      - IMG_20260306_151957_168.jpg: terlalu gelap (brightness=7.8)
      - IMG_20260306_151944_507.jpg: terlalu gelap (brightness=9.3)
      - IMG_20260306_151940_562.jpg: terlalu gelap (brightness=9.3)
      - IMG_20260306_152244_931.jpg: blur berat (

---
## 🖼️ Step 5 — Standarisasi Format

Semua gambar dikonversi ke format standar yang siap digunakan untuk training model:

| Parameter | Nilai |
|-----------|-------|
| Format | JPG |
| Ukuran | 224 × 224 px |
| Channel | RGB |
| Normalisasi | [0, 1] |
| Resampling | Lanczos |


In [ ]:
def standardize_and_save(stats, dup_report, quality_report, out_dir) -> dict:
    """Standarisasi gambar: format, ukuran, channel, dan normalisasi."""
    print("=" * 55)
    print("  STEP 5: STANDARISASI")
    print("=" * 55)

    # Kumpulkan gambar yang perlu dilewati
    skipped_paths = set()
    for cls in CLASSES:
        for p, _ in dup_report[cls]["exact_duplicates"]:
            skipped_paths.add(p)
        for p, _ in quality_report[cls]:
            skipped_paths.add(p)

    processed_stats = {}
    for cls in CLASSES:
        src_paths  = stats[cls]["paths"]
        dst_dir    = Path(out_dir) / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        saved       = 0
        skipped     = 0
        norm_arrays = []

        for p in src_paths:
            if p in skipped_paths:
                skipped += 1
                continue
            try:
                with Image.open(p) as img:
                    img = img.convert(COLOR_MODE)
                    img = img.resize(TARGET_SIZE, Image.LANCZOS)

                    out_path = dst_dir / f"{p.stem}{TARGET_FORMAT}"
                    img.save(out_path, quality=95)

                    arr = np.array(img, dtype=np.float32)
                    norm_arrays.append(arr / 255.0 if NORM_RANGE == (0,1)
                                       else (arr / 127.5) - 1.0)
                    saved += 1
            except Exception as e:
                print(f"  ⚠️  Gagal proses {p.name}: {e}")
                skipped += 1

        processed_stats[cls] = {"saved": saved, "skipped": skipped}

        if norm_arrays:
            all_arr = np.stack(norm_arrays)
            print(f"  ✅ Kelas '{cls}': {saved} disimpan, {skipped} dilewati")
            print(f"     Normalisasi → [{all_arr.min():.3f}, {all_arr.max():.3f}]  (target: {NORM_RANGE})")
            print(f"     Ukuran: {TARGET_SIZE} | Channel: {COLOR_MODE} | Format: {TARGET_FORMAT.upper()}")

    return processed_stats


processed_stats = standardize_and_save(init_stats, dup_report, quality_report, PROCESSED_DIR)


---
## 🔄 Step 6 — Augmentasi Data

Setiap gambar asli menghasilkan **6 variasi augmentasi** menggunakan kombinasi acak dari 8 teknik:

| # | Teknik | Parameter |
|---|--------|-----------|
| 1 | **Rotasi** | ±20° |
| 2 | **Horizontal Flip** | 50% peluang |
| 3 | **Vertical Flip** | 25% peluang |
| 4 | **Zoom** | ±15% |
| 5 | **Crop Tepi** | 10% dari tepi |
| 6 | **Brightness** | 0.7 – 1.3× |
| 7 | **Contrast** | 0.7 – 1.3× |
| 8 | **Sharpness** | 0.5 – 2.0× (25% peluang) |


In [ ]:
import random

def augment_image(img: Image.Image, aug_id: int) -> Image.Image:
    """Terapkan kombinasi augmentasi acak pada satu gambar."""
    rng = random.Random(aug_id * 1337)

    # 1. Rotasi
    img = img.rotate(rng.uniform(-ROTATION_RANGE, ROTATION_RANGE),
                     resample=Image.BILINEAR, expand=False)
    # 2. Horizontal flip
    if rng.random() > 0.5:
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
    # 3. Vertical flip
    if rng.random() > 0.75:
        img = img.transpose(Image.FLIP_TOP_BOTTOM)
    # 4. Zoom (crop + resize)
    zoom = rng.uniform(0, ZOOM_RANGE)
    w, h = img.size
    mx, my = int(w * zoom), int(h * zoom)
    if mx > 0 and my > 0:
        img = img.crop((mx, my, w - mx, h - my))
        img = img.resize(TARGET_SIZE, Image.LANCZOS)
    # 5. Crop tepi
    cp = int(min(img.size) * CROP_FRACTION)
    w, h = img.size
    img = img.crop((cp, cp, w - cp, h - cp))
    img = img.resize(TARGET_SIZE, Image.LANCZOS)
    # 6. Brightness
    img = ImageEnhance.Brightness(img).enhance(rng.uniform(*BRIGHTNESS_RANGE))
    # 7. Contrast
    img = ImageEnhance.Contrast(img).enhance(rng.uniform(*CONTRAST_RANGE))
    # 8. Sharpness
    if rng.random() > 0.75:
        img = ImageEnhance.Sharpness(img).enhance(rng.uniform(0.5, 2.0))

    return img


def augment_dataset(processed_dir: str, aug_dir: str) -> dict:
    """Augmentasi seluruh dataset dan simpan ke folder output."""
    print("=" * 55)
    print("  STEP 6: AUGMENTASI DATASET")
    print("=" * 55)
    print(f"  Teknik : Rotasi(±{ROTATION_RANGE}°) · HFlip · VFlip · Zoom · Crop · Brightness · Contrast · Sharpness")
    print(f"  Jumlah : {AUG_PER_IMAGE}× per gambar")

    aug_stats = {}
    for cls in CLASSES:
        src_dir   = Path(processed_dir) / cls
        dst_dir   = Path(aug_dir) / cls
        dst_dir.mkdir(parents=True, exist_ok=True)

        images    = list(src_dir.glob(f"*{TARGET_FORMAT}"))
        aug_count = 0

        for img_path in images:
            shutil.copy2(img_path, dst_dir / img_path.name)
            try:
                with Image.open(img_path) as img:
                    img_rgb = img.convert(COLOR_MODE)
                    for j in range(AUG_PER_IMAGE):
                        aug_img  = augment_image(img_rgb.copy(), j)
                        aug_name = f"{img_path.stem}_aug{j+1:02d}{TARGET_FORMAT}"
                        aug_img.save(dst_dir / aug_name, quality=95)
                        aug_count += 1
            except Exception as e:
                print(f"  ⚠️  Gagal augmentasi {img_path.name}: {e}")

        total = len(images) + aug_count
        aug_stats[cls] = {"original": len(images), "augmented": aug_count, "total": total}
        print(f"  ✅ Kelas '{cls}':")
        print(f"     Gambar asli    : {len(images)}")
        print(f"     Hasil augmentasi: {aug_count}")
        print(f"     Total          : {total}")

    return aug_stats


aug_stats = augment_dataset(PROCESSED_DIR, AUGMENTED_DIR)


  STEP 6: AUGMENTASI DATASET
  Teknik : Rotasi(±20°) · HFlip · VFlip · Zoom · Crop · Brightness · Contrast · Sharpness
  Jumlah : 6× per gambar
  ✅ Kelas 'daun_bagus':
     Gambar asli    : 56
     Hasil augmentasi: 336
     Total          : 392
  ✅ Kelas 'daun_layu':
     Gambar asli    : 48
     Hasil augmentasi: 288
     Total          : 336


---
## 📊 Step 7 — Laporan Visual

Visualisasi statistik hasil preprocessing untuk kebutuhan laporan akademis.


In [ ]:
# ── Data statistik ──────────────────────────────────────────────
init_counts      = {"daun_bagus": 64,  "daun_layu": 64}
removed_counts   = {"daun_bagus": 8,   "daun_layu": 16}
processed_counts = {"daun_bagus": 56,  "daun_layu": 48}
aug_counts       = {"daun_bagus": 336, "daun_layu": 288}
total_counts     = {"daun_bagus": 392, "daun_layu": 336}
blur_removed     = {"daun_bagus": 8,   "daun_layu": 12}
dark_removed     = {"daun_bagus": 0,   "daun_layu": 4}

# ── Styling ──────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family"   : "DejaVu Sans",
    "axes.spines.top"   : False,
    "axes.spines.right" : False,
    "axes.grid"     : True,
    "grid.alpha"    : 0.3,
    "grid.linestyle": "--",
})

GREEN  = "#2C5F2D"
MOSS   = "#97BC62"
BLUE   = "#2471A3"
ORANGE = "#E67E22"
RED    = "#C0392B"
TEAL   = "#1ABC9C"

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.patch.set_facecolor("#FAFAFA")
fig.suptitle("Laporan Preprocessing & Augmentasi Dataset Daun",
             fontsize=15, fontweight="bold", color="#1A2E1A", y=1.03)

x = np.arange(len(CLASSES))
lbl = ["Daun Bagus", "Daun Layu"]

# ── Plot 1: Sebelum vs Sesudah Cleaning ──────────────────────────
ax = axes[0]
ax.set_facecolor("#F8FBF8")
b1 = ax.bar(x - 0.2, [init_counts[c] for c in CLASSES],      0.38, label="Sebelum", color=BLUE,  alpha=0.85)
b2 = ax.bar(x + 0.2, [processed_counts[c] for c in CLASSES], 0.38, label="Sesudah", color=GREEN, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbl)
ax.set_title("Sebelum vs Sesudah Cleaning", fontweight="bold", pad=10)
ax.set_ylabel("Jumlah Gambar")
ax.legend(framealpha=0.7)
for i, c in enumerate(CLASSES):
    ax.text(i-0.2, init_counts[c]+0.8,      str(init_counts[c]),      ha="center", fontweight="bold", fontsize=10)
    ax.text(i+0.2, processed_counts[c]+0.8, str(processed_counts[c]), ha="center", fontweight="bold", fontsize=10)

# ── Plot 2: Gambar Dihapus per Alasan ────────────────────────────
ax = axes[1]
ax.set_facecolor("#FBF8F8")
b3 = ax.bar(x, [blur_removed[c] for c in CLASSES], 0.45, label="Blur Berat",    color=RED,    alpha=0.85)
b4 = ax.bar(x, [dark_removed[c] for c in CLASSES],
            0.45, bottom=[blur_removed[c] for c in CLASSES],
            label="Terlalu Gelap", color=ORANGE, alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbl)
ax.set_title("Gambar Dihapus per Alasan", fontweight="bold", pad=10)
ax.set_ylabel("Jumlah")
ax.legend(framealpha=0.7)
for i, c in enumerate(CLASSES):
    total_rm = removed_counts[c]
    ax.text(i, total_rm+0.3, str(total_rm), ha="center", fontweight="bold", fontsize=11)

# ── Plot 3: Hasil Augmentasi ─────────────────────────────────────
ax = axes[2]
ax.set_facecolor("#F8FBFB")
orig = [processed_counts[c] for c in CLASSES]
augs = [aug_counts[c] for c in CLASSES]
b5 = ax.bar(x, orig, 0.5, label="Gambar Asli",   color=TEAL,   alpha=0.85)
b6 = ax.bar(x, augs, 0.5, bottom=orig, label="Augmentasi", color=MOSS,   alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(lbl)
ax.set_title(f"Hasil Augmentasi (×{AUG_PER_IMAGE} per gambar)", fontweight="bold", pad=10)
ax.set_ylabel("Jumlah Gambar")
ax.legend(framealpha=0.7)
for i, c in enumerate(CLASSES):
    ax.text(i, total_counts[c]+4,
            f"Total\n{total_counts[c]}", ha="center", fontweight="bold", fontsize=9)

plt.tight_layout()
plt.savefig("preprocessing_report.png", dpi=150, bbox_inches="tight",
            facecolor="#FAFAFA")
plt.show()
print("✅ Laporan disimpan: preprocessing_report.png")


---
## ✅ Ringkasan Akhir

Semua tahap preprocessing telah selesai dijalankan.


In [ ]:
print("╔══════════════════════════════════════════════════════╗")
print("║           RINGKASAN AKHIR PREPROCESSING              ║")
print("╠══════════════════════════════════════════════════════╣")

for cls in CLASSES:
    dupes_exact = len(dup_report[cls]["exact_duplicates"])
    dupes_near  = len(dup_report[cls]["near_duplicates"])
    issues      = len(quality_report[cls])
    saved       = processed_stats[cls]["saved"]
    total_aug   = aug_stats[cls]["total"]

    print(f"║  [{cls}]")
    print(f"║    Gambar awal        : {init_stats[cls]['total']}")
    print(f"║    Duplikat ditemukan : {dupes_exact} persis + {dupes_near} hampir sama")
    print(f"║    Gambar bermasalah  : {issues}")
    print(f"║    Setelah cleaning   : {saved}")
    print(f"║    Setelah augmentasi : {total_aug}")
    print("║")

print(f"║  📁 Output cleaning    → {PROCESSED_DIR}/")
print(f"║  📁 Output augmentasi  → {AUGMENTED_DIR}/")
print(f"║  📄 Laporan visual     → preprocessing_report.png")
print("╚══════════════════════════════════════════════════════╝")


╔══════════════════════════════════════════════════════╗
║           RINGKASAN AKHIR PREPROCESSING              ║
╠══════════════════════════════════════════════════════╣
║  [daun_bagus]
║    Gambar awal        : 64
║    Duplikat ditemukan : 0 persis + 0 hampir sama
║    Gambar bermasalah  : 8
║    Setelah cleaning   : 56
║    Setelah augmentasi : 392
║
║  [daun_layu]
║    Gambar awal        : 64
║    Duplikat ditemukan : 0 persis + 0 hampir sama
║    Gambar bermasalah  : 16
║    Setelah cleaning   : 48
║    Setelah augmentasi : 336
║
║  📁 Output cleaning    → dataset_processed/
║  📁 Output augmentasi  → dataset_augmented/
║  📄 Laporan visual     → preprocessing_report.png
╚══════════════════════════════════════════════════════╝


---

<div style="background:#f0f7f0; border-left:4px solid #2C5F2D; padding:14px 18px; border-radius:0 8px 8px 0; margin-top:8px;">
  <b style="color:#2C5F2D;">📌 Catatan Penting</b><br>
  <ul style="margin:8px 0 0 0; padding-left:20px; color:#333; font-size:13px;">
    <li><b>Augmentasi hanya pada training set</b> — validasi set menggunakan gambar asli untuk evaluasi yang objektif.</li>
    <li><b>Split data</b> — gunakan <code>train_test_split(stratify=y)</code> dengan rasio <b>80% train / 20% val</b>.</li>
    <li><b>Threshold blur</b> disesuaikan dari default 50.0 → 5.0 karena gambar daun memiliki tekstur halus (Laplacian variance rendah secara natural).</li>
  </ul>
</div>

<div style="text-align:center; color:#999; font-size:12px; margin-top:20px;">
  Preprocessing Dataset Klasifikasi Daun &nbsp;|&nbsp; 2026
</div>
